In [ ]:
from dotenv import load_dotenv

load_dotenv(r"", override=True)

True

In [ ]:
# ==============================================================================
# BLOCK 1: IMPORTS & CONFIGURATION
# ==============================================================================
import os
import re
import json
import sqlite3
from datetime import datetime, timezone
from typing import TypedDict, List, Dict, Any, Literal, Optional
from pydantic import BaseModel, Field

import httpx
import ollama
from dotenv import load_dotenv
from google import genai
from google.genai import types
from langgraph.graph import StateGraph, START, END

# Load your api.env file (adjust path if moved)
load_dotenv(r"C:\Users\hari7\Documents\Browser_agent_hackathon\Agentware_demo\Hari\api.env", override=True)

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
KEENABLE_API_KEY = os.environ["KEENABLE_API_KEY"]
KEENABLE_BASE_URL = "https://api.keenable.ai"

# Supervisor tries these Gemini models in order, then falls back to local Qwen 4B
SUPERVISOR_GEMINI_CHAIN = [
    "gemini-3.8-flash",
    "gemini-3.5-flash",
    "gemini-2.5-flash",
]
WORKER_MODEL = "qwen3.5:4b"     # Official Qwen 3.5 4B (fast with think=False)
WORKER_THINK = False            # Disables slow thinking tokens while keeping native tool parsing
DB_PATH = "agent_context.db"
OLLAMA_OPTIONS = {}             # Set to {"num_gpu": 0} if Windows CUDA crashes

gemini_client = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None


# ==============================================================================
# BLOCK 2: SQLITE + FTS5 CONTEXT-OFFLOADING DATABASE + QUICK-FACT EXTRACTION
# ==============================================================================
def init_db(db_path: str = DB_PATH) -> None:
    with sqlite3.connect(db_path) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS pages (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                search_title TEXT NOT NULL,
                site TEXT,
                url TEXT NOT NULL,
                title TEXT,
                content TEXT NOT NULL,
                fetched_at TEXT NOT NULL
            )
        """)
        conn.execute("""
            CREATE VIRTUAL TABLE IF NOT EXISTS pages_fts USING fts5(
                search_title, url, title, content,
                content='pages', content_rowid='id'
            )
        """)
        conn.execute("""
            CREATE TRIGGER IF NOT EXISTS pages_ai AFTER INSERT ON pages BEGIN
                INSERT INTO pages_fts(rowid, search_title, url, title, content)
                VALUES (new.id, new.search_title, new.url, new.title, new.content);
            END;
        """)
        conn.commit()


def save_pages_to_db(search_title: str, pages: List[Dict[str, Any]], db_path: str = DB_PATH) -> str:
    now = datetime.now(timezone.utc).isoformat()
    saved_count = 0
    with sqlite3.connect(db_path) as conn:
        conn.execute("DELETE FROM pages WHERE search_title = ?", (search_title,))
        conn.execute("INSERT INTO pages_fts(pages_fts) VALUES('rebuild')")
        for p in pages:
            content = p.get("content", "")
            if not content:
                continue
            conn.execute(
                "INSERT INTO pages (search_title, site, url, title, content, fetched_at) VALUES (?, ?, ?, ?, ?, ?)",
                (search_title, p.get("site", ""), p.get("url", ""), p.get("title", "Untitled"), content, now),
            )
            saved_count += 1
        conn.commit()
    return (
        f"Saved {saved_count} fresh page(s) in '{db_path}' under search_title='{search_title}'. "
        f"Ready for db_analyzer to query via query_saved_search()."
    )


def extract_quick_facts(search_title: str, db_path: str = DB_PATH) -> str:
    """Deterministically pulls URL, title, ₹ prices, and X.X ratings from saved SQLite pages."""
    with sqlite3.connect(db_path) as conn:
        rows = conn.execute(
            "SELECT site, title, url, content FROM pages WHERE search_title = ?",
            (search_title,),
        ).fetchall()

    if not rows:
        return "No saved pages found."

    lines = []
    for site, title, url, content in rows:
        prices = list(dict.fromkeys(re.findall(r"₹\s?[\d,]+(?:\.\d{2})?", content)))[:4]
        ratings = list(dict.fromkeys(re.findall(r"\b[1-5]\.\d\b(?=\s*(?:out of 5|stars?|★|ratings?))", content, re.I)))[:2]
        lines.append(
            f"- [{site or 'web'}] {title}\n"
            f"  URL: {url}\n"
            f"  Detected Prices: {prices or 'None in indexed text'}\n"
            f"  Detected Ratings: {ratings or 'None in indexed text'}"
        )
    return "\n".join(lines)


def print_db_preview(search_title: str, limit: int = 3, db_path: str = DB_PATH) -> None:
    """Prints a compact preview of rows currently stored in SQLite for `search_title`."""
    print(f"\n--- [DB PREVIEW: {db_path} | search_title='{search_title}'] ---")
    with sqlite3.connect(db_path) as conn:
        rows = conn.execute(
            """
            SELECT id, site, title, url, length(content), substr(content, 1, 140)
            FROM pages
            WHERE search_title = ?
            ORDER BY id DESC
            LIMIT ?
            """,
            (search_title, limit),
        ).fetchall()

    if not rows:
        print("  (No rows found)")
        return
    for row_id, site, title, url, char_len, snippet in rows:
        clean_snip = snippet.replace("\n", " ").strip()
        print(f"  Row #{row_id} | site={site or 'n/a'} | chars={char_len:,}")
        print(f"    Title  : {title}")
        print(f"    URL    : {url}")
        print(f"    Excerpt: {clean_snip}...")
    print("-" * 60)


def query_saved_search(search_title: str, keyword: str, limit: int = 6) -> str:
    """
    TOOL FOR WORKER: Returns compact FTS5 snippets matching `keyword`.
    Falls back to SQL LIKE when `keyword` contains symbols (like '₹') that FTS5 ignores.
    """
    raw_kw = str(keyword or "").strip()
    if not raw_kw:
        return "Empty keyword. Pass a specific word like 'price', 'rating', 'cotton', or 'RTX'."

    cleaned = re.sub(r'["\'*():^]', " ", raw_kw).strip()
    has_alnum = bool(re.search(r"[A-Za-z0-9]", cleaned))

    rows = []
    if has_alnum:
        try:
            with sqlite3.connect(DB_PATH) as conn:
                rows = conn.execute(
                    """
                    SELECT url, title, snippet(pages_fts, 3, '[', ']', ' ... ', 30) AS excerpt
                    FROM pages_fts
                    WHERE search_title = ? AND pages_fts MATCH ?
                    ORDER BY rank
                    LIMIT ?
                    """,
                    (search_title, f'"{cleaned}"', limit),
                ).fetchall()
        except sqlite3.OperationalError:
            rows = []

    # Fallback: SQL LIKE search for symbols (e.g. '₹') or when FTS5 returns nothing
    if not rows:
        with sqlite3.connect(DB_PATH) as conn:
            like_rows = conn.execute(
                """
                SELECT url, title, content
                FROM pages
                WHERE search_title = ? AND content LIKE ?
                LIMIT ?
                """,
                (search_title, f"%{raw_kw}%", limit),
            ).fetchall()

        for url, title, content in like_rows:
            idx = content.lower().find(raw_kw.lower())
            start = max(0, idx - 80)
            end = min(len(content), idx + 160)
            excerpt = content[start:end].replace("\n", " ").strip()
            rows.append((url, title, f"... {excerpt} ..."))

    if not rows:
        return f"No matches for '{raw_kw}' under search_title='{search_title}'."
    return "\n\n".join(f"- Title: {t}\n  URL: {u}\n  Match: {e}" for u, t, e in rows)


# ==============================================================================
# BLOCK 3: KEENABLE API WRAPPERS (SEARCH + BATCH FETCH)
# ==============================================================================
def keenable_search(query: str, site: str = "", max_results: int = 3) -> List[Dict[str, Any]]:
    payload: Dict[str, Any] = {
        "query": query,
        "max_results": max_results,
        "snippet_max_length": 250,
        "mode": "realtime",
    }
    if site:
        payload["site"] = site
    resp = httpx.post(
        f"{KEENABLE_BASE_URL}/v1/search",
        headers={"X-API-Key": KEENABLE_API_KEY, "Content-Type": "application/json"},
        json=payload,
        timeout=25.0,
    )
    resp.raise_for_status()
    return resp.json().get("results", [])


def keenable_fetch_batch(urls: List[str], max_chars: int = 15000) -> List[Dict[str, Any]]:
    if not urls:
        return []
    resp = httpx.post(
        f"{KEENABLE_BASE_URL}/v1/fetch",
        headers={"X-API-Key": KEENABLE_API_KEY, "Content-Type": "application/json"},
        json={"urls": urls[:20], "content": {"max_chars": max_chars}},
        timeout=35.0,
    )
    resp.raise_for_status()
    return resp.json().get("results", [])


# ==============================================================================
# BLOCK 4: SUPERVISOR SYSTEM MANIFEST & PYDANTIC ROUTING SCHEMA
# ==============================================================================
SUPERVISOR_SYSTEM_PROMPT = f"""
You are the Supervisor orchestrating a shopping research pipeline.
You understand every worker, its model, its tools, and where data is stored:

STORAGE ARCHITECTURE:
- Database: Local SQLite file '{DB_PATH}' with table `pages` and FTS5 index `pages_fts`.
- Key: All fetched product Markdown is stored and queried under `search_title`.

AVAILABLE WORKERS:
1. "see_first"
   - Model: {WORKER_MODEL} (local 4B, think={WORKER_THINK})
   - Tools: Keenable Search (`POST /v1/search`, compact 250-char snippets)
   - Job: Runs lightweight recon across `search_queries` and `target_sites`, inspects snippets,
          and sets `proceed_to_fetch` + `selected_urls`. Run this FIRST after creating the plan.

2. "fetch_and_offload"
   - Model: None (deterministic Python)
   - Tools: Keenable Batch Fetch (`POST /v1/fetch`) + SQLite writer (`save_pages_to_db`) + regex price/rating extractor
   - Job: Fetches full clean Markdown for `selected_urls` and offloads it into '{DB_PATH}'
          under `search_title`. Run this when `proceed_to_fetch` is True and `db_receipt` is empty.

3. "db_analyzer"
   - Model: {WORKER_MODEL} (local 4B, think={WORKER_THINK})
   - Tools: SQLite FTS5 reader (`query_saved_search(search_title, keyword)`)
   - Job: Queries '{DB_PATH}' for prices, ratings, and specs, then writes `final_answer`.
          Run this AFTER `fetch_and_offload` has saved pages to SQLite.

4. "answer_from_snippets"
   - Model: {WORKER_MODEL} (local 4B, think={WORKER_THINK})
   - Tools: None (reads `recon_results` directly)
   - Job: Early-exit synthesis ONLY if `see_first` found that snippets already answer the query
          (`proceed_to_fetch` is False).

5. "finish"
   - Job: End the workflow ONLY when `has_final_answer` is True.

RULES:
- On step 1 (`plan_created` is False), populate `plan` and route `next_worker` to "see_first".
- Never route to "finish" if `has_final_answer` is False.
"""


class SearchPlan(BaseModel):
    search_title: str = Field(description="Short snake_case label for SQLite storage (e.g. 'red_cotton_shirt_men').")
    search_queries: List[str] = Field(description="One concise search query per distinct item requested.")
    target_sites: List[str] = Field(description="Target shopping domains (default ['amazon.in', 'flipkart.com']).")
    constraints: str = Field(description="Budget, preferred colors, or required specs extracted from the user prompt.")


class SupervisorDecision(BaseModel):
    next_worker: Literal["see_first", "fetch_and_offload", "db_analyzer", "answer_from_snippets", "finish"]
    task_instruction: str = Field(description="Specific instruction for the chosen worker.")
    reasoning: str = Field(description="1-sentence explanation of this routing decision.")
    plan: Optional[SearchPlan] = Field(
        default=None,
        description="Required on step 1 to break down the user request; omit or null on later steps.",
    )


# ==============================================================================
# BLOCK 5: SHARED LANGGRAPH STATE, SUPERVISOR CASCADE & WORKER NODES
# ==============================================================================
class AgentState(TypedDict):
    user_query: str
    next_worker: str
    task_instruction: str
    step_count: int
    max_steps: int
    plan_created: bool
    search_title: str
    search_queries: List[str]
    target_sites: List[str]
    constraints: str
    recon_results: List[Dict[str, Any]]
    proceed_to_fetch: bool
    selected_urls: List[str]
    db_receipt: str
    final_answer: str
    history: List[str]
    route_trace: List[str]


def _call_supervisor_with_fallback(state_snapshot: Dict[str, Any]) -> tuple[SupervisorDecision, str]:
    """Tries Gemini 3.8 -> 3.5 -> 2.5 Flash, then falls back to local Qwen 3.5 4B."""
    prompt = f"Current Pipeline State:\n{json.dumps(state_snapshot, indent=2)}\n\nDecide the next step."

    if gemini_client:
        for model_name in SUPERVISOR_GEMINI_CHAIN:
            try:
                response = gemini_client.models.generate_content(
                    model=model_name,
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        system_instruction=SUPERVISOR_SYSTEM_PROMPT,
                        response_mime_type="application/json",
                        response_schema=SupervisorDecision,
                        temperature=0.1,
                    ),
                )
                return SupervisorDecision.model_validate_json(response.text), model_name
            except Exception as e:
                print(f"   [FALLBACK CHECK] {model_name} failed ({type(e).__name__}) -> trying next...")

    print(f"   [FALLBACK CHECK] Using local {WORKER_MODEL} as Supervisor...")
    res = ollama.chat(
        model=WORKER_MODEL,
        format=SupervisorDecision.model_json_schema(),
        think=WORKER_THINK,
        options=OLLAMA_OPTIONS,
        keep_alive=0,
        messages=[
            {"role": "system", "content": SUPERVISOR_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
    )
    return SupervisorDecision.model_validate_json(res.message.content), f"local:{WORKER_MODEL}"


def supervisor_node(state: AgentState) -> Dict[str, Any]:
    step_num = state["step_count"] + 1
    print(f"\n==================== [STEP {step_num}: SUPERVISOR CHECK] ====================")

    if state["step_count"] >= state["max_steps"]:
        print("[CHECK] Max steps reached -> forcing route to 'finish'")
        return {"next_worker": "finish", "route_trace": state["route_trace"] + ["supervisor -> finish (max_steps)"]}

    state_snapshot = {
        "user_query": state["user_query"],
        "plan_created": state["plan_created"],
        "search_title": state["search_title"],
        "search_queries": state["search_queries"],
        "target_sites": state["target_sites"],
        "constraints": state["constraints"],
        "recon_count": len(state["recon_results"]),
        "proceed_to_fetch": state["proceed_to_fetch"],
        "selected_urls_count": len(state["selected_urls"]),
        "db_receipt": state["db_receipt"],
        "has_final_answer": bool(state["final_answer"].strip()),
        "history": state["history"],
    }
    print(f"[STATE CHECK] plan={state_snapshot['plan_created']} | recon={state_snapshot['recon_count']} | "
          f"proceed={state_snapshot['proceed_to_fetch']} | urls={state_snapshot['selected_urls_count']} | "
          f"has_db={bool(state_snapshot['db_receipt'])} | has_answer={state_snapshot['has_final_answer']}")

    decision, used_model = _call_supervisor_with_fallback(state_snapshot)
    print(f"[SUPERVISOR ({used_model})] Decision -> {decision.next_worker}")
    print(f"[REASONING] {decision.reasoning}")
    print(f"[INSTRUCTION] {decision.task_instruction}")

    updates: Dict[str, Any] = {
        "next_worker": decision.next_worker,
        "task_instruction": decision.task_instruction,
        "step_count": step_num,
        "history": state["history"] + [f"Supervisor({used_model}) -> {decision.next_worker}: {decision.reasoning}"],
        "route_trace": state["route_trace"] + [f"supervisor({used_model}) -> {decision.next_worker}"],
    }

    if decision.plan and not state["plan_created"]:
        updates.update({
            "plan_created": True,
            "search_title": decision.plan.search_title,
            "search_queries": decision.plan.search_queries,
            "target_sites": decision.plan.target_sites or ["amazon.in", "flipkart.com"],
            "constraints": decision.plan.constraints,
        })
        print(f"[PLAN BREAKDOWN] title='{decision.plan.search_title}' | queries={decision.plan.search_queries} | "
              f"sites={updates['target_sites']} | constraints='{decision.plan.constraints}'")

    return updates


def see_first_worker(state: AgentState) -> Dict[str, Any]:
    print("\n--- [WORKER: see_first] ---")
    sites = state["target_sites"] or ["amazon.in", "flipkart.com"]
    queries = state["search_queries"] or [state["user_query"]]
    max_urls = min(max(3, len(queries) * 2), 8)
    per_site_hits = 1 if len(queries) >= 3 else 2

    all_results = []
    for q in queries:
        for site in sites:
            hits = keenable_search(q, site=site, max_results=per_site_hits)
            print(f"  [KEENABLE SEARCH] site='{site}' | query='{q}' -> {len(hits)} hit(s)")
            for h in hits:
                h["site"] = site
                h["item_query"] = q
                all_results.append(h)

    preview = [
        {"item": r.get("item_query"), "url": r.get("url"), "title": r.get("title"), "snippet": r.get("snippet", "")}
        for r in all_results
    ]

    res = ollama.chat(
        model=WORKER_MODEL,
        format="json",
        think=WORKER_THINK,
        options=OLLAMA_OPTIONS,
        keep_alive=0,
        messages=[
            {
                "role": "system",
                "content": (
                    f"You are a triage worker. Select up to {max_urls} best product page URLs covering all requested items.\n"
                    'Return JSON with exact keys: "proceed_to_fetch" (bool), "selected_urls" (list of URLs), "reason" (str).'
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Supervisor instruction: {state['task_instruction']}\n"
                    f"Constraints: {state['constraints']}\n\n"
                    f"Search snippets:\n{json.dumps(preview, indent=2)}"
                ),
            },
        ],
    )
    decision = json.loads(res.message.content)
    selected = decision.get("selected_urls", [])[:max_urls]
    print(f"  [TRIAGE CHECK] proceed_to_fetch={decision.get('proceed_to_fetch')} | selected={len(selected)} URL(s)")
    print(f"  [TRIAGE REASON] {decision.get('reason')}")
    return {
        "recon_results": all_results,
        "proceed_to_fetch": bool(decision.get("proceed_to_fetch", False)),
        "selected_urls": selected,
        "history": state["history"] + [f"see_first: {len(all_results)} snippets -> selected {len(selected)} URLs"],
    }


def fetch_and_offload_worker(state: AgentState) -> Dict[str, Any]:
    print("\n--- [WORKER: fetch_and_offload] ---")
    urls = state["selected_urls"]
    print(f"  [KEENABLE FETCH] Batch-fetching {len(urls)} URL(s)...")
    pages = keenable_fetch_batch(urls, max_chars=15000)

    for p in pages:
        url = p.get("url", "")
        p["site"] = "amazon.in" if "amazon" in url else ("flipkart.com" if "flipkart" in url else "")

    base_receipt = save_pages_to_db(state["search_title"], pages)
    quick_facts = extract_quick_facts(state["search_title"])
    full_receipt = f"{base_receipt}\n\nQuick Facts Extracted:\n{quick_facts}"

    print(f"  [SQLITE WRITE] {base_receipt}")
    print(f"\n--- [QUICK FACTS EXTRACTED] ---\n{quick_facts}")
    print_db_preview(state["search_title"], limit=3)

    return {
        "db_receipt": full_receipt,
        "history": state["history"] + [f"fetch_and_offload: {base_receipt}"],
    }


def db_analyzer_worker(state: AgentState) -> Dict[str, Any]:
    print("\n--- [WORKER: db_analyzer] ---")
    messages = [
        {
            "role": "system",
            "content": (
                "You are a research analyst. Full webpage text is stored in SQLite and key detected prices/ratings "
                "are already listed in the Database status. Call `query_saved_search` with 1–3 targeted keywords "
                "(e.g. 'cotton', 'fit', 'rating', 'RTX', 'RAM') to gather extra specs, then stop calling tools."
            ),
        },
        {
            "role": "user",
            "content": (
                f"User question: {state['user_query']}\n"
                f"Supervisor instruction: {state['task_instruction']}\n"
                f"Constraints: {state['constraints']}\n"
                f"Database status:\n{state['db_receipt']}\n\n"
                f"search_title: {state['search_title']}"
            ),
        },
    ]

    for step in range(4):
        res = ollama.chat(
            model=WORKER_MODEL,
            messages=messages,
            tools=[query_saved_search],
            think=WORKER_THINK,
            options=OLLAMA_OPTIONS,
            keep_alive=0,
        )
        messages.append(res.message)

        if not res.message.tool_calls:
            answer = (res.message.content or "").strip()
            if answer:
                print(f"  [ANALYZER] Finished early on step {step + 1} ({len(answer)} chars).")
                return {
                    "final_answer": answer,
                    "history": state["history"] + ["db_analyzer: synthesized final answer from SQLite"],
                }
            break

        for call in res.message.tool_calls:
            args = call.function.arguments
            print(f"  [DB TOOL CALL {step + 1}] query_saved_search({args})")
            tool_output = query_saved_search(
                search_title=args.get("search_title", state["search_title"]),
                keyword=args.get("keyword", ""),
            )
            messages.append({"role": "tool", "content": tool_output})

    print("  [ANALYZER] Running final synthesis (tools disabled)...")
    messages.append({
        "role": "user",
        "content": (
            "Stop calling tools. Using ONLY the Quick Facts and database snippets retrieved above, "
            "write the final Markdown comparison table (Product, Platform, Price, Rating, Key Details) "
            "and a concise recommendation now."
        ),
    })
    final_res = ollama.chat(
        model=WORKER_MODEL,
        messages=messages,
        think=WORKER_THINK,
        options=OLLAMA_OPTIONS,
        keep_alive=0,
    )
    answer = (final_res.message.content or "").strip() or "Could not synthesize answer from retrieved snippets."
    print(f"  [ANALYZER] Final answer generated ({len(answer)} chars).")
    return {
        "final_answer": answer,
        "history": state["history"] + ["db_analyzer: synthesized final answer after DB lookups"],
    }


def answer_from_snippets_worker(state: AgentState) -> Dict[str, Any]:
    print("\n--- [WORKER: answer_from_snippets] ---")
    res = ollama.chat(
        model=WORKER_MODEL,
        think=WORKER_THINK,
        options=OLLAMA_OPTIONS,
        keep_alive=0,
        messages=[
            {"role": "system", "content": "Answer the user concisely using ONLY the provided search snippets."},
            {"role": "user", "content": f"Query: {state['user_query']}\n\nSnippets:\n{json.dumps(state['recon_results'], indent=2)}"},
        ],
    )
    answer = (res.message.content or "").strip() or "No answer could be generated from snippets."
    return {
        "final_answer": answer,
        "history": state["history"] + ["answer_from_snippets: answered directly from snippets"],
    }


# ==============================================================================
# BLOCK 6: BUILD & COMPILE THE SUPERVISOR-WORKER GRAPH
# ==============================================================================
def route_from_supervisor(state: AgentState) -> str:
    target = state["next_worker"]
    print(f"[GRAPH ROUTE] supervisor ===> {target}")
    return target


init_db()

workflow = StateGraph(AgentState)
workflow.add_node("supervisor", supervisor_node)
workflow.add_node("see_first", see_first_worker)
workflow.add_node("fetch_and_offload", fetch_and_offload_worker)
workflow.add_node("db_analyzer", db_analyzer_worker)
workflow.add_node("answer_from_snippets", answer_from_snippets_worker)

workflow.add_edge(START, "supervisor")
workflow.add_conditional_edges(
    "supervisor",
    route_from_supervisor,
    {
        "see_first": "see_first",
        "fetch_and_offload": "fetch_and_offload",
        "db_analyzer": "db_analyzer",
        "answer_from_snippets": "answer_from_snippets",
        "finish": END,
    },
)

for worker_name in ["see_first", "fetch_and_offload", "db_analyzer", "answer_from_snippets"]:
    workflow.add_edge(worker_name, "supervisor")

agent = workflow.compile()


# ==============================================================================
# BLOCK 7: RUN THE SUPERVISOR-WORKER PIPELINE & PRINT FLOW TRACE
# ==============================================================================
result = agent.invoke({
    "user_query": "Find a red cotton shirt for men on amazon.in and flipkart.com and compare prices and ratings",
    "next_worker": "",
    "task_instruction": "",
    "step_count": 0,
    "max_steps": 8,
    "plan_created": False,
    "search_title": "",
    "search_queries": [],
    "target_sites": [],
    "constraints": "",
    "recon_results": [],
    "proceed_to_fetch": False,
    "selected_urls": [],
    "db_receipt": "",
    "final_answer": "",
    "history": [],
    "route_trace": [],
})

print("\n" + "=" * 60)
print("EXECUTION ROUTE TRACE")
print("=" * 60)
for idx, hop in enumerate(result["route_trace"], start=1):
    print(f"  {idx}. {hop}")

print("\n" + "=" * 60)
print("FINAL ANSWER")
print("=" * 60)
print(result["final_answer"])


==================== [STEP 1: SUPERVISOR CHECK] ====================
[STATE CHECK] plan=False | recon=0 | proceed=False | urls=0 | has_db=False | has_answer=False
[SUPERVISOR (gemini-3.8-flash)] Decision -> see_first
[REASONING] Initializing the research plan and routing to see_first to begin product discovery.
[INSTRUCTION] Execute search across target sites for red cotton shirts for men, evaluate snippets, and determine URLs to fetch for price and rating comparison.
[PLAN BREAKDOWN] title='red_cotton_shirt_men' | queries=['red cotton shirt men'] | sites=['amazon.in', 'flipkart.com'] | constraints='Men's red cotton shirt; compare prices and ratings across amazon.in and flipkart.com.'
[GRAPH ROUTE] supervisor ===> see_first

--- [WORKER: see_first] ---
  [KEENABLE SEARCH] site='amazon.in' | query='red cotton shirt men' -> 2 hit(s)
  [KEENABLE SEARCH] site='flipkart.com' | query='red cotton shirt men' -> 2 hit(s)
  [TRIAGE CHECK] proceed_to_fetch=True | selected=3 URL(s)
  [TRIAGE REAS